# RoadSafe India — Data Cleaning

## Objective

The purpose of this notebook is to transform the raw RoadSafe India
road-accident dataset into a clean, validated and analysis-ready dataset.

The original dataset stored in `data/raw/` will not be modified.

The cleaned dataset will be saved in `data/processed/`.

## Cleaning Workflow

1. Load the raw dataset
2. Inspect the data
3. Standardize column names
4. Clean text fields
5. Convert numeric columns
6. Check missing values
7. Check duplicate records
8. Validate numerical values
9. Validate State/UT uniqueness
10. Save the processed dataset

In [1]:
import pandas as pd

In [2]:
# Load the original raw dataset
raw_path = "../data/raw/state_wise_road_accidents_2020_2024.csv"

df = pd.read_csv(raw_path)

print("Raw dataset loaded successfully.")
print(f"Shape: {df.shape}")

Raw dataset loaded successfully.
Shape: (39, 14)


In [3]:
# Create a separate working copy.
# The original raw data remains untouched.

clean_df = df.copy()

print("Working copy created.")

Working copy created.


In [4]:
# Standardize column names

clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace("%", "percent", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print("Cleaned column names:")
print(clean_df.columns.tolist())

Cleaned column names:
['sl_no', 'state', '2020_accidents', '2021_accidents', '2022_accidents', '2023_accidents', '2024_accidents', 'change_from_2023_to_2024', 'percent_change_from_2023_to_2024', '2020_ranking', '2021_ranking', '2022_ranking', '2023_ranking', '2024_ranking']


In [5]:
# Clean whitespace around State/UT names

clean_df["state"] = (
    clean_df["state"]
    .astype("string")
    .str.strip()
)

print("State/UT values after text cleaning:")
display(clean_df["state"])

State/UT values after text cleaning:


0                           Andhra Pradesh
1                        Arunachal Pradesh
2                                    Assam
3                                    Bihar
4                             Chhattisgarh
5                                      Goa
6                                  Gujarat
7                                  Haryana
8                         Himachal Pradesh
9                                Jharkhand
10                               Karnataka
11                                  Kerala
12                          Madhya Pradesh
13                             Maharashtra
14                                 Manipur
15                               Meghalaya
16                                 Mizoram
17                                Nagaland
18                                  Odisha
19                                  Punjab
20                               Rajasthan
21                                  Sikkim
22                              Tamil Nadu
23         

In [6]:
# Identify columns that should contain numerical values

numeric_columns = [
    column
    for column in clean_df.columns
    if column != "state"
]

print("Columns expected to contain numerical values:")
for column in numeric_columns:
    print("-", column)

Columns expected to contain numerical values:
- sl_no
- 2020_accidents
- 2021_accidents
- 2022_accidents
- 2023_accidents
- 2024_accidents
- change_from_2023_to_2024
- percent_change_from_2023_to_2024
- 2020_ranking
- 2021_ranking
- 2022_ranking
- 2023_ranking
- 2024_ranking


In [7]:
# Convert all non-State columns to numeric values.
# Invalid values are converted to NaN so they can be detected.

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

print("Numeric conversion completed.")

Numeric conversion completed.


In [8]:
# Check for missing values after conversion

missing_values = clean_df.isnull().sum()

print("Missing values:")
display(missing_values)

print(f"\nTotal missing values: {missing_values.sum()}")

Missing values:


sl_no                                3
state                                2
2020_accidents                      27
2021_accidents                      27
2022_accidents                      27
2023_accidents                      27
2024_accidents                      27
change_from_2023_to_2024             7
percent_change_from_2023_to_2024     2
2020_ranking                         4
2021_ranking                         4
2022_ranking                         3
2023_ranking                         3
2024_ranking                         5
dtype: int64


Total missing values: 168


In [9]:
# Check for completely duplicated rows

duplicate_rows = clean_df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 1


In [10]:
# Check whether a State/UT appears more than once

duplicate_states = clean_df["state"].duplicated().sum()

print(f"Duplicate State/UT names: {duplicate_states}")

if duplicate_states > 0:
    print("\nDuplicated State/UT entries:")
    display(
        clean_df[
            clean_df["state"].duplicated(keep=False)
        ].sort_values("state")
    )

Duplicate State/UT names: 1

Duplicated State/UT entries:


,sl_no,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,change_from_2023_to_2024,percent_change_from_2023_to_2024,2020_ranking,2021_ranking,2022_ranking,2023_ranking,2024_ranking
37,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Identify accident-count columns

accident_columns = [
    "2020_accidents",
    "2021_accidents",
    "2022_accidents",
    "2023_accidents",
    "2024_accidents"
]

print("Minimum accident count:")
display(clean_df[accident_columns].min())

print("\nNegative accident-count values:")

negative_accident_values = clean_df[
    (clean_df[accident_columns] < 0).any(axis=1)
]

if negative_accident_values.empty:
    print("No negative accident counts found.")
else:
    display(negative_accident_values)

Minimum accident count:


2020_accidents    1.0
2021_accidents    4.0
2022_accidents    3.0
2023_accidents    1.0
2024_accidents    0.0
dtype: float64


Negative accident-count values:
No negative accident counts found.


In [12]:
# Check for empty or missing State/UT names

invalid_states = clean_df[
    clean_df["state"].isna() |
    (clean_df["state"].str.strip() == "")
]

print(f"Invalid or missing State/UT entries: {len(invalid_states)}")

if not invalid_states.empty:
    display(invalid_states)

Invalid or missing State/UT entries: 2


,sl_no,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,change_from_2023_to_2024,percent_change_from_2023_to_2024,2020_ranking,2021_ranking,2022_ranking,2023_ranking,2024_ranking
37,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Final summary of the cleaned DataFrame

print("CLEANING SUMMARY")
print("----------------")

print(f"Rows: {clean_df.shape[0]}")
print(f"Columns: {clean_df.shape[1]}")
print(f"Missing values: {clean_df.isnull().sum().sum()}")
print(f"Duplicate rows: {clean_df.duplicated().sum()}")
print(f"Unique State/UTs: {clean_df['state'].nunique()}")

CLEANING SUMMARY
----------------
Rows: 39
Columns: 14
Missing values: 168
Duplicate rows: 1
Unique State/UTs: 37


In [14]:
# Display the cleaned dataset

display(clean_df.head())

,sl_no,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,change_from_2023_to_2024,percent_change_from_2023_to_2024,2020_ranking,2021_ranking,2022_ranking,2023_ranking,2024_ranking
0,1.0,Andhra Pradesh,NaN,NaN,NaN,NaN,NaN,-392.0,-2.0,7.0,7.0,9.0,9.0,9.0
1,2.0,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,-10.0,-3.5,32.0,28.0,31.0,29.0,27.0
2,3.0,Assam,NaN,NaN,NaN,NaN,NaN,427.0,5.8,16.0,16.0,16.0,16.0,16.0
3,4.0,Bihar,NaN,NaN,NaN,NaN,NaN,596.0,5.4,15.0,15.0,14.0,14.0,14.0
4,5.0,Chhattisgarh,NaN,NaN,NaN,NaN,NaN,NaN,10.3,11.0,11.0,12.0,12.0,11.0


In [15]:
# Save the cleaned dataset

processed_path = "../data/processed/state_wise_road_accidents_2020_2024_cleaned.csv"

clean_df.to_csv(processed_path, index=False)

print("Cleaned dataset saved successfully.")
print(f"Location: {processed_path}")

Cleaned dataset saved successfully.
Location: ../data/processed/state_wise_road_accidents_2020_2024_cleaned.csv


In [16]:
# Load the processed file again to verify that it was saved correctly

verification_df = pd.read_csv(processed_path)

print("Processed file verification")
print("---------------------------")
print(f"Shape: {verification_df.shape}")

display(verification_df.head())

Processed file verification
---------------------------
Shape: (39, 14)


,sl_no,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,change_from_2023_to_2024,percent_change_from_2023_to_2024,2020_ranking,2021_ranking,2022_ranking,2023_ranking,2024_ranking
0,1.0,Andhra Pradesh,NaN,NaN,NaN,NaN,NaN,-392.0,-2.0,7.0,7.0,9.0,9.0,9.0
1,2.0,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,-10.0,-3.5,32.0,28.0,31.0,29.0,27.0
2,3.0,Assam,NaN,NaN,NaN,NaN,NaN,427.0,5.8,16.0,16.0,16.0,16.0,16.0
3,4.0,Bihar,NaN,NaN,NaN,NaN,NaN,596.0,5.4,15.0,15.0,14.0,14.0,14.0
4,5.0,Chhattisgarh,NaN,NaN,NaN,NaN,NaN,NaN,10.3,11.0,11.0,12.0,12.0,11.0


## Cleaning Conclusions

The raw State/UT-level road-accident dataset was inspected and transformed
into an analysis-ready dataset.

The original raw dataset was preserved in `data/raw/`.

The cleaned dataset was saved in `data/processed/`.

The cleaned dataset will be used for subsequent exploratory data analysis.